# DAPT-01: Jain Domain Adaptation Training

## Qwen3-8B-Base → QLoRA Domain Adaptation

This notebook executes the DAPT-01 experiment on Kaggle GPU.

**Configuration:**
- Base model: Qwen/Qwen3-8B-Base
- Method: QLoRA
- LoRA rank: 16
- Learning rate: 2e-4
- Epochs: 1
- Batch size: 2
- Gradient accumulation: 8
- Max sequence length: 2048

## 1. Environment Verification

In [ ]:
!nvidia-smi

import torch
import sys
import os

print(f"Python: {sys.version}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    print(f"VRAM: {props.total_memory / 1024**3:.2f} GB")
    print(f"CUDA version: {torch.version.cuda}")
else:
    raise RuntimeError("No GPU available. This notebook requires GPU runtime.")

## 2. Install Dependencies

In [ ]:
!pip install -q transformers peft accelerate bitsandbytes datasets safetensors tensorboard

## 3. Import Libraries

In [ ]:
import json
import time
import yaml
from pathlib import Path
from datetime import datetime

import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import load_dataset, Dataset, DatasetDict
import bitsandbytes as bnb

print("All imports successful")

## 4. Configuration

In [ ]:
CONFIG = {
    "model_name": "Qwen/Qwen3-8B-Base",
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "learning_rate": 2e-4,
    "epochs": 1,
    "batch_size": 1,
    "gradient_accumulation": 16,
    "max_seq_length": 2048,
    "seed": 42,
    "output_dir": "/kaggle/working/dapt-01-output",
    "dataset_path": "/kaggle/input/jain-domain-train/",
    "checkpoint_path": "/kaggle/input/dapt01-checkpoint-3500/",
    "checkpoint_step": 3500,
    "session_timeout_minutes": 120,
    "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
}

print(json.dumps(CONFIG, indent=2))

## 5. Find Dataset Path

In [ ]:
# Discover the actual dataset path on Kaggle
import glob

# List all files under /kaggle/input/
input_files = glob.glob("/kaggle/input/**/*.jsonl", recursive=True)
print("Found JSONL files:")
for f in input_files:
    print(f"  {f}")

# Update CONFIG with the correct path
if input_files:
    # Find the directory containing train.jsonl
    for f in input_files:
        if "train.jsonl" in f:
            CONFIG["dataset_path"] = str(Path(f).parent)
            break
    print(f"\nDataset path: {CONFIG['dataset_path']}")
else:
    # Fallback: check the expected path
    expected = "/kaggle/input/jain-domain-train/"
    if os.path.exists(expected):
        CONFIG["dataset_path"] = expected
    else:
        print(f"WARNING: No dataset found. Expected at {expected}")
        print("Please add the dataset as a data source in the notebook settings.")

## 6. Dataset Verification

In [ ]:
# Count records in each split
for split in ["train.jsonl", "val.jsonl", "test.jsonl"]:
    path = Path(CONFIG["dataset_path"]) / split
    count = sum(1 for _ in open(path))
    print(f"{split}: {count} records")

# Verify first record has required fields
with open(Path(CONFIG["dataset_path"]) / "train.jsonl") as f:
    first = json.loads(f.readline())
    print(f"\nFields: {list(first.keys())}")
    assert "text" in first, "Missing 'text' field"
    assert "provenance" in first, "Missing 'provenance' field"
    print("Dataset verification: PASS")

## 7. Load Dataset

In [ ]:
def load_jsonl_text_only(file_path):
    """Load JSONL file and extract only the text field."""
    texts = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                record = json.loads(line)
                if 'text' in record and record['text']:
                    texts.append(record['text'])
    return texts

train_texts = load_jsonl_text_only(Path(CONFIG["dataset_path"]) / "train.jsonl")
val_texts = load_jsonl_text_only(Path(CONFIG["dataset_path"]) / "val.jsonl")

dataset = DatasetDict({
    "train": Dataset.from_dict({"text": train_texts}),
    "validation": Dataset.from_dict({"text": val_texts}),
})
print(f"Train: {len(dataset['train'])}, Validation: {len(dataset['validation'])}")

## 8. Load Tokenizer and Model

In [ ]:
print(f"Loading tokenizer: {CONFIG['model_name']}")
tokenizer = AutoTokenizer.from_pretrained(
    CONFIG["model_name"],
    trust_remote_code=True,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print(f"Tokenizer loaded. Vocab size: {tokenizer.vocab_size}")

print(f"Loading model: {CONFIG['model_name']}")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    CONFIG["model_name"],
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
print(f"Model loaded. Parameters: {model.num_parameters():,}")

## 9. Apply LoRA

In [ ]:
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=CONFIG["lora_r"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=CONFIG["lora_dropout"],
    target_modules=CONFIG["target_modules"],
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 10. Baseline Evaluation

In [ ]:
def evaluate_model(model, tokenizer, test_prompts, max_new_tokens=200):
    """Evaluate model on test prompts."""
    results = []
    model.eval()
    with torch.no_grad():
        for prompt in test_prompts:
            inputs = tokenizer(prompt["prompt"], return_tensors="pt").to(model.device)
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                temperature=0.7,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id,
            )
            response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
            results.append({
                "category": prompt["category"],
                "prompt": prompt["prompt"],
                "response": response,
                "expected_keywords": prompt.get("keywords", []),
            })
    return results

# Evaluation prompts (defined for later sessions, not run now)
EVAL_PROMPTS = [
    {"category": "jain_terminology", "prompt": "What is Ahimsa in Jainism?", "keywords": ["ahimsa", "non-violence", "jain"]},
    {"category": "jain_terminology", "prompt": "Explain the concept of Anekantavada.", "keywords": ["anekantavada", "many-sidedness"]},
    {"category": "jain_terminology", "prompt": "What is the significance of Navkar Mantra?", "keywords": ["navkar", "mantra", "jain"]},
    {"category": "jain_terminology", "prompt": "What are the Five Mahavratas in Jainism?", "keywords": ["mahavratas", "five", "vow"]},
    {"category": "jain_terminology", "prompt": "Explain Sallekhana in Jain tradition.", "keywords": ["sallekhana", "fasting", "death"]},
    {"category": "sect_isolation", "prompt": "What is the difference between Sthanakavasi and Murtipujaka traditions?", "keywords": ["sthanakavasi", "murtipujaka"]},
    {"category": "sect_isolation", "prompt": "How do Digambara and Shvetambara Jains differ?", "keywords": ["digambara", "shvetambara"]},
    {"category": "sth TERMINOLOGY", "prompt": "What is the Sthanakavasi approach to worship?", "keywords": ["sthanakavasi", "sthana", "imageless"]},
    {"category": "prakrit", "prompt": "What language is used in Jain Agamas?", "keywords": ["prakrit", "ardhamagadhi", "agama"]},
    {"category": "hindi", "prompt": "जैन धर्म में अहिंसा का क्या महत्व है?", "keywords": ["ahinsa", "jain", "mahatva"]},
]

# SKIP baseline evaluation this session (task 55c: no eval, save GPU time)
baseline_results = []
baseline_dir = Path(CONFIG["output_dir"]) / "baseline"
print("Baseline evaluation SKIPPED this session (no eval per task 55c)")

## 11. Tokenize Dataset

In [ ]:
class TextDataCollator:
    """Custom collator that tokenizes raw text on-the-fly."""
    def __init__(self, tokenizer, max_length=2048):
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __call__(self, batch):
        texts = [item["text"] for item in batch]
        tokenized = self.tokenizer(
            texts,
            truncation=True,
            padding="longest",
            max_length=self.max_length,
            return_tensors="pt",
        )
        tokenized["labels"] = tokenized["input_ids"].clone()
        return tokenized

data_collator = TextDataCollator(tokenizer, max_length=CONFIG["max_seq_length"])

print("Using custom TextDataCollator for on-the-fly tokenization")
print(f"Train: {len(dataset['train'])}, Validation: {len(dataset['validation'])}")

## 12. DAPT-01 Training

In [ ]:
os.makedirs(CONFIG["output_dir"], exist_ok=True)

# Copy checkpoint to output dir for resume
import shutil
ckpt_src = CONFIG["checkpoint_path"]
ckpt_dst = os.path.join(CONFIG["output_dir"], f"checkpoint-{CONFIG['checkpoint_step']}")
if os.path.exists(ckpt_src) and not os.path.exists(ckpt_dst):
    shutil.copytree(ckpt_src, ckpt_dst)
    print(f"Copied checkpoint to {ckpt_dst}")
else:
    print(f"Checkpoint already exists or source not found")

# Time safety: track session start to stop gracefully before Kaggle timeout
import time
from transformers import TrainerCallback
SESSION_START = time.time()
SESSION_TIMEOUT_SEC = CONFIG["session_timeout_minutes"] * 60

class TimeSafetyCallback(TrainerCallback):
    def on_step_end(self, args, state, control, **kwargs):
        elapsed = time.time() - SESSION_START
        if elapsed > SESSION_TIMEOUT_SEC:
            control.should_training_stop = True
            print(f"\n[TIME SAFETY] Stopping at step {state.global_step} after {elapsed/60:.1f} min (limit {CONFIG['session_timeout_minutes']} min). Last checkpoint will be preserved.")
        return control

print(f"Session timeout: {CONFIG['session_timeout_minutes']} min ({SESSION_TIMEOUT_SEC/3600:.2f} h)")


training_args = TrainingArguments(
    output_dir=CONFIG["output_dir"],
    num_train_epochs=CONFIG["epochs"],
    per_device_train_batch_size=CONFIG["batch_size"],
    per_device_eval_batch_size=CONFIG["batch_size"],
    gradient_accumulation_steps=CONFIG["gradient_accumulation"],
    learning_rate=CONFIG["learning_rate"],
    weight_decay=0.01,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    max_grad_norm=1.0,
    logging_steps=10,
    save_steps=5000,
    eval_steps=5000,
    save_total_limit=3,
    fp16=True,
    bf16=False,
    dataloader_num_workers=2,
    seed=CONFIG["seed"],
    report_to="tensorboard",
    run_name="dapt-01-qwen3-8b-jain-domain",
    eval_strategy="no",
    load_best_model_at_end=False,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    gradient_checkpointing=True,
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    data_collator=data_collator,
    processing_class=tokenizer,
    callbacks=[TimeSafetyCallback()],
)

print("Starting DAPT-01 training...")
print(f"Start time: {datetime.now().isoformat()}")
start_time = time.time()

# === STATUS REPORT ===
import torch
print("\n=== DAPT-01 RESUME SESSION STATUS ===")
print(f"GPU detected: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Model: {CONFIG['model_name']}")
print(f"Resume checkpoint: {ckpt_dst}")
print(f"Detected global step: {CONFIG['checkpoint_step']}")
print(f"Target global step: 15248")
print(f"Dataset: {CONFIG['dataset_path']} ({len(dataset['train'])} train / {len(dataset['validation'])} val)")
print(f"Config: LoRA r={CONFIG['lora_r']} a={CONFIG['lora_alpha']} lr={CONFIG['learning_rate']} batch={CONFIG['batch_size']}x{CONFIG['gradient_accumulation']} seq={CONFIG['max_seq_length']} fp16")
print(f"Eval during training: DISABLED")
print(f"Session timeout: {CONFIG['session_timeout_minutes']} min ({SESSION_TIMEOUT_SEC/3600:.2f} h)")
print(f"Est. throughput: ~17 s/step → ~{int(SESSION_TIMEOUT_SEC/17)} steps this session")
print("=====================================\n")

train_result = trainer.train(resume_from_checkpoint=True)

elapsed = time.time() - start_time
print(f"\nTraining complete in {elapsed/3600:.2f} hours")
print(f"End time: {datetime.now().isoformat()}")

## 13. Save Checkpoint

In [ ]:
# Save final model
trainer.save_model()
tokenizer.save_pretrained(CONFIG["output_dir"])

# Save training metrics
metrics = train_result.metrics
with open(Path(CONFIG["output_dir"]) / "metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

# Save config
with open(Path(CONFIG["output_dir"]) / "config.json", "w") as f:
    json.dump(CONFIG, f, indent=2)

# === TIME-SAFE RESUMABLE CHECKPOINT ===
# If training stopped before reaching a save_steps boundary, create a resumable
# checkpoint-<final_step> so the next session can resume from the exact state.
final_step = trainer.state.global_step
final_ckpt = Path(CONFIG["output_dir"]) / f"checkpoint-{final_step}"
if not final_ckpt.exists():
    print(f"Creating resumable checkpoint at step {final_step}...")
    final_ckpt.mkdir(parents=True, exist_ok=True)
    trainer.save_model(str(final_ckpt))
    tokenizer.save_pretrained(str(final_ckpt))
    if trainer.optimizer is not None:
        torch.save(trainer.optimizer.state_dict(), final_ckpt / "optimizer.pt")
    if trainer.lr_scheduler is not None:
        torch.save(trainer.lr_scheduler.state_dict(), final_ckpt / "scheduler.pt")
    trainer.state.save_to_json(final_ckpt / "trainer_state.json")
    torch.save(trainer.args, final_ckpt / "training_args.bin")
    print(f"Resumable checkpoint saved: {final_ckpt}")
else:
    print(f"Checkpoint already exists at step {final_step}: {final_ckpt}")

# Verify integrity of the newest checkpoint
required = ["adapter_model.safetensors", "optimizer.pt", "scheduler.pt", "trainer_state.json", "tokenizer.json"]
missing = [f for f in required if not (final_ckpt / f).exists()]
if missing:
    print(f"WARNING: checkpoint-{final_step} missing: {missing}")
else:
    print(f"Integrity verified: checkpoint-{final_step} complete")

print(f"\nFinal global step: {final_step} / 15248 ({final_step/15248*100:.1f}%)")
print(f"Next resume checkpoint: checkpoint-{final_step}")
print(f"Checkpoint saved to {CONFIG['output_dir']}")
print(f"Training loss: {metrics.get('train_loss', 'N/A')}")
print(f"Training runtime: {metrics.get('train_runtime', 'N/A')} seconds")

## 14. DAPT-01 Evaluation

In [ ]:
# SKIP DAPT evaluation this session (task 55c: no eval per instructions)
dapt_results = []
eval_dir = Path(CONFIG["output_dir"]) / "evaluation"
print("DAPT-01 evaluation SKIPPED this session (no eval per task 55c)")

## 15. Base vs DAPT Comparison

In [ ]:
# SKIP Base vs DAPT comparison this session (task 55c: no eval per instructions)
print("Base vs DAPT comparison SKIPPED this session (no eval per task 55c)")

## 16. Export Artifacts

In [ ]:
# Create artifact manifest
manifest = {
    "experiment_id": "DAPT-01",
    "date": datetime.now().isoformat(),
    "status": "COMPLETE",
    "model": CONFIG["model_name"],
    "training_backend": "KAGGLE_GPU",
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A",
    "vram_gb": round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2) if torch.cuda.is_available() else 0,
    "config": CONFIG,
    "training_metrics": metrics,
    "artifacts": {
        "checkpoint": str(Path(CONFIG["output_dir"]) / "adapter_model"),
        "baseline": str(baseline_dir),
        "evaluation": str(eval_dir),
    },
}

with open(Path(CONFIG["output_dir"]) / "manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

print("\n" + "=" * 80)
print("DAPT-01 COMPLETE")
print("=" * 80)
print(f"Output: {CONFIG['output_dir']}")
print(f"Training loss: {metrics.get('train_loss', 'N/A')}")
print(f"Training time: {elapsed/3600:.2f} hours")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")